In [1]:

import sqlite3
from pathlib import Path

# Find the SQL script
possible_paths = [
    Path("sql/01_create_tables.sql"),
    Path("../sql/01_create_tables.sql")
]

sql_path = next(
    (p for p in possible_paths if p.exists()),
    None
)

if sql_path is None:
    raise FileNotFoundError("SQL script not found")

# Locate the project folder
project_root = sql_path.parent.parent

# Choose where to save our database
db_path = (
    project_root / "data" / "processed"
    / "service_performance.db"
)

# Connect to SQLite
conn = sqlite3.connect(db_path)

# Enable foreign key checks
conn.execute("PRAGMA foreign_keys = ON")

# Read and execute our SQL script
sql_script = sql_path.read_text(encoding="utf-8")

conn.executescript(sql_script)

# Verify that the tables exist
tables = [
    "DimDate",
    "DimCentre",
    "FactServicePerformance"
]

for table in tables:
    count = conn.execute(
        f"SELECT COUNT(*) FROM {table}"
    ).fetchone()[0]

    print(f"{table}: {count} records")

conn.close()

print("\nDatabase created successfully!")
print("Database location:", db_path)


DimDate: 0 records
DimCentre: 0 records
FactServicePerformance: 0 records

Database created successfully!
Database location: ../data/processed/service_performance.db


In [2]:

import pandas as pd

# STEP 16: Load dimension tables

# Read the analysis-ready CSV
csv_path = (
    db_path.parent /
    "csc_wait_times_analysis_ready.csv"
)

data = pd.read_csv(csv_path)

# Convert Date from text to datetime
data["Date"] = pd.to_datetime(
    data["Date"],
    format="%Y-%m-%d"
)

# -------- CREATE DATE DIMENSION --------

date_dim = data[["Date"]].drop_duplicates().copy()

date_dim = date_dim.sort_values("Date")

date_dim["DateKey"] = (
    date_dim["Date"].dt.strftime("%Y%m%d").astype(int)
)

date_dim["FullDate"] = (
    date_dim["Date"].dt.strftime("%Y-%m-%d")
)

date_dim["Year"] = date_dim["Date"].dt.year

date_dim["Month"] = date_dim["Date"].dt.month

date_dim["DayOfWeek"] = (
    date_dim["Date"].dt.day_name()
)

print("DATE DIMENSION:")
display(date_dim.head())


# -------- CREATE CENTRE DIMENSION --------

centre_dim = data[
    ["CSC_NAME", "CENTRE_CATEGORY"]
].drop_duplicates().sort_values(
    "CSC_NAME"
).reset_index(drop=True)

# Confirm each centre has one category
assert len(centre_dim) == data["CSC_NAME"].nunique()

# Assign a unique ID to every centre
centre_dim.insert(
    0,
    "CentreKey",
    range(1, len(centre_dim) + 1)
)

print("\nSERVICE CENTRE DIMENSION:")
display(centre_dim.head())


# -------- INSERT INTO SQLITE --------

conn = sqlite3.connect(db_path)

conn.execute("PRAGMA foreign_keys = ON")

with conn:

    conn.executemany(
        """INSERT OR IGNORE INTO DimDate
        (DateKey, FullDate, Year, Month, DayOfWeek)
        VALUES (?, ?, ?, ?, ?)""",

        list(date_dim[
            ["DateKey", "FullDate", "Year",
             "Month", "DayOfWeek"]
        ].itertuples(index=False, name=None))
    )

    conn.executemany(
        """INSERT OR IGNORE INTO DimCentre
        (CentreKey, CentreName, ReportingCategory)
        VALUES (?, ?, ?)""",

        list(centre_dim[
            ["CentreKey", "CSC_NAME", "CENTRE_CATEGORY"]
        ].itertuples(index=False, name=None))
    )


# -------- VERIFY TABLE COUNTS --------

print("\nDATABASE RECORD COUNTS:")

for table in ["DimDate", "DimCentre"]:

    count = conn.execute(
        f"SELECT COUNT(*) FROM {table}"
    ).fetchone()[0]

    print(table, ":", count)

conn.close()


DATE DIMENSION:


,Date,DateKey,FullDate,Year,Month,DayOfWeek
0,2018-07-15,20180715,2018-07-15,2018,7,Sunday
1,2018-07-16,20180716,2018-07-16,2018,7,Monday
2,2018-07-17,20180717,2018-07-17,2018,7,Tuesday
4,2018-07-18,20180718,2018-07-18,2018,7,Wednesday
7,2018-07-19,20180719,2018-07-19,2018,7,Thursday



SERVICE CENTRE DIMENSION:


,CentreKey,CSC_NAME,CENTRE_CATEGORY
0,1,Atherton CSC,All zero
1,2,Ayr QGAP,Mixed
2,3,Barcaldine CSC,All zero
3,4,Beaudesert CSC,Mixed
4,5,Beenleigh CSC,No zeros



DATABASE RECORD COUNTS:
DimDate : 1800
DimCentre : 99


In [3]:

# STEP 17: Load the FactServicePerformance table

# Reconnect to our SQLite database
conn = sqlite3.connect(db_path)
conn.execute("PRAGMA foreign_keys = ON")

# Get the service centre IDs from DimCentre
centre_lookup = pd.read_sql_query(
    "SELECT CentreKey, CentreName FROM DimCentre",
    conn
)

# Create a mapping: Centre Name -> CentreKey
centre_map = dict(zip(
    centre_lookup["CentreName"],
    centre_lookup["CentreKey"]
))

# Prepare the fact table
fact_data = pd.DataFrame({
    "DateKey": data["Date"].dt.strftime("%Y%m%d").astype(int),
    "CentreKey": data["CSC_NAME"].map(centre_map),
    "Customers": data["CUSTOMERS"].astype(int),
    "WaitMinutes": data["WAIT_MINUTES"].astype(float),
    "ZeroWaitFlag": data["ZERO_WAIT_FLAG"].astype(int),
    "LongWaitFlag": data["LONG_WAIT_FLAG"].astype(int)
})

# Validate relationships before inserting
assert fact_data["CentreKey"].notna().all()

fact_data["CentreKey"] = fact_data["CentreKey"].astype(int)

assert not fact_data.duplicated(
    subset=["DateKey", "CentreKey"]
).any()

print("FACT TABLE PREVIEW:")
display(fact_data.head())

# Insert records into SQLite
# Rerunning updates existing centre-date records
insert_sql = """
INSERT INTO FactServicePerformance
(DateKey, CentreKey, Customers, WaitMinutes,
 ZeroWaitFlag, LongWaitFlag)
VALUES (?, ?, ?, ?, ?, ?)
ON CONFLICT(DateKey, CentreKey) DO UPDATE SET
    Customers = excluded.Customers,
    WaitMinutes = excluded.WaitMinutes,
    ZeroWaitFlag = excluded.ZeroWaitFlag,
    LongWaitFlag = excluded.LongWaitFlag
"""

with conn:
    conn.executemany(
        insert_sql,
        list(fact_data.itertuples(index=False, name=None))
    )

# Verify the database results
summary = pd.read_sql_query("""
SELECT
    COUNT(*) AS TotalRecords,
    SUM(Customers) AS TotalCustomers,
    SUM(ZeroWaitFlag) AS ZeroWaitRecords,
    SUM(LongWaitFlag) AS LongWaitRecords
FROM FactServicePerformance
""", conn)

print("\nDATABASE VALIDATION RESULTS:")
display(summary)

assert summary["TotalRecords"].iloc[0] == len(data)
assert summary["TotalCustomers"].iloc[0] == data["CUSTOMERS"].sum()
assert summary["ZeroWaitRecords"].iloc[0] == 77277
assert summary["LongWaitRecords"].iloc[0] == 30

# Check foreign key integrity
assert conn.execute(
    "PRAGMA foreign_key_check"
).fetchall() == []

print("\nFACT TABLE LOADED AND VALIDATED!")

conn.close()


FACT TABLE PREVIEW:


,DateKey,CentreKey,Customers,WaitMinutes,ZeroWaitFlag,LongWaitFlag
0,20180715,48,7,730.666667,0,1
1,20180716,48,363,12.033333,0,0
2,20180717,48,343,7.883333,0,0
3,20180717,90,5,186.833333,0,1
4,20180718,48,282,10.200000,0,0



DATABASE VALIDATION RESULTS:


,TotalRecords,TotalCustomers,ZeroWaitRecords,LongWaitRecords
0,146219,22575820,77277,30



FACT TABLE LOADED AND VALIDATED!


In [4]:

# STEP 18: Execute our first business SQL query

# Connect to our database
conn = sqlite3.connect(db_path)

# Read SQL from the saved file
query_path = project_root / "sql" / "02_business_queries.sql"

sql_query = query_path.read_text(encoding="utf-8")

# Execute the SQL and return results as a DataFrame
result = pd.read_sql_query(sql_query, conn)

print("TOP 10 SERVICE CENTRES BY CUSTOMER VOLUME")

display(result)

# Close connection
conn.close()


TOP 10 SERVICE CENTRES BY CUSTOMER VOLUME


,CentreName,ReportingCategory,RecordedDays,TotalCustomerVisits,ZeroWaitDays,ZeroWaitPercentage
0,Logan City CSC,Mixed,1728,969219,7,0.41
1,Macgregor CSC,No zeros,1677,774673,0,0.00
2,Helensvale CSC,Mixed,1645,721566,2,0.12
3,Greenslopes CSC,Mixed,1662,625796,14,0.84
4,Chermside CSC,No zeros,1649,610838,0,0.00
5,Maroochydore CSC,Mixed,1660,594952,8,0.48
6,Caboolture CSC,No zeros,1665,586290,0,0.00
7,Southport CSC,Mixed,1669,583215,17,1.02
8,Burleigh Waters CSC,Mixed,1645,564450,1,0.06
9,Cairns CSC,No zeros,1672,548982,0,0.00


In [6]:

# STEP 19: Validate BQ1 using SQL

conn = sqlite3.connect(db_path)

query_path = (
    project_root / "sql" /
    "03_bq1_total_customer_visits.sql"
)

sql_query = query_path.read_text(encoding="utf-8")

bq1_result = pd.read_sql_query(sql_query, conn)

print("BQ1: OVERALL CUSTOMER VISIT PERFORMANCE")
display(bq1_result)

# Reconcile against the source dataset
assert bq1_result["TotalCustomerVisits"].iloc[0] == (
    data["CUSTOMERS"].sum()
)

assert bq1_result["TotalRecords"].iloc[0] == len(data)

print("SQL AND SOURCE DATA RECONCILIATION PASSED!")

conn.close()


BQ1: OVERALL CUSTOMER VISIT PERFORMANCE


,TotalRecords,TotalCentres,RecordedDates,TotalCustomerVisits
0,146219,99,1800,22575820


SQL AND SOURCE DATA RECONCILIATION PASSED!


In [7]:

# STEP 20: BQ2 - Waiting Time Analysis

conn = sqlite3.connect(db_path)

query_path = (
    project_root / "sql" /
    "04_bq2_waiting_time.sql"
)

sql_query = query_path.read_text(encoding="utf-8")

bq2_result = pd.read_sql_query(sql_query, conn)

print("BQ2: CUSTOMER WAITING TIME ANALYSIS")

display(bq2_result)

# Confirm basic counts match our dataset
assert bq2_result["TotalRecords"].iloc[0] == len(data)

assert bq2_result["TotalVisits"].iloc[0] == (
    data["CUSTOMERS"].sum()
)

print("BQ2 BASIC RECONCILIATION PASSED!")

conn.close()


BQ2: CUSTOMER WAITING TIME ANALYSIS


,TotalRecords,TotalVisits,ZeroWaitRecords,WeightedWait_AllRecords,WeightedWait_PositiveOnly,ZeroWait_VisitPercentage
0,146219,22575820,77277,9.68,11.46,15.55


BQ2 BASIC RECONCILIATION PASSED!


In [8]:

# STEP 21: BQ3 - Service Centre Wait Comparison

conn = sqlite3.connect(db_path)

query_path = (
    project_root / "sql" /
    "05_bq3_centre_wait_comparison.sql"
)

sql_query = query_path.read_text(encoding="utf-8")

bq3_result = pd.read_sql_query(sql_query, conn)

print("BQ3: TOP 10 CENTRES BY WEIGHTED WAITING TIME")

display(bq3_result)

conn.close()


BQ3: TOP 10 CENTRES BY WEIGHTED WAITING TIME


,CentreName,ReportingCategory,RecordedDays,TotalCustomerVisits,WeightedWaitMinutes,LongWaitDays
0,Chermside CSC,No zeros,1649,610838,16.38,0
1,Cairns CSC,No zeros,1672,548982,15.87,2
2,Bethania CSC,No zeros,333,86726,15.73,0
3,Townsville Garbutt CSC,No zeros,1680,452415,15.58,0
4,Caboolture CSC,No zeros,1665,586290,15.40,1
5,Strathpine CSC,No zeros,1646,514785,14.77,0
6,Beenleigh CSC,No zeros,1216,449663,13.58,0
7,Toowoomba Harristown CSC,No zeros,1725,448612,11.80,0
8,Beenleigh CSC 2,No zeros,391,79304,11.57,1
9,Bundamba CSC,No zeros,643,288559,10.82,3


In [9]:

# STEP 22: BQ4 - Monthly Customer Demand

conn = sqlite3.connect(db_path)

query_path = (
    project_root / "sql" /
    "06_bq4_monthly_demand.sql"
)

sql_query = query_path.read_text(encoding="utf-8")

bq4_result = pd.read_sql_query(sql_query, conn)

print("BQ4: MONTHLY CUSTOMER DEMAND")
display(bq4_result.head(12))

print("\nTOP 10 MONTHS BY AVERAGE VISITS PER CENTRE-DAY")

display(
    bq4_result.sort_values(
        "AvgVisitsPerCentreDay",
        ascending=False
    ).head(10)
)

# Check monthly totals against our original data
assert bq4_result["TotalCustomerVisits"].sum() == (
    data["CUSTOMERS"].sum()
)

print("\nMONTHLY CUSTOMER TOTALS RECONCILED!")

conn.close()


BQ4: MONTHLY CUSTOMER DEMAND


,Year,Month,YearMonth,TotalCustomerVisits,RecordedCentreDays,RecordedDates,ReportingCentres,AvgVisitsPerCentreDay
0,2018,7,2018-07,13689,50,13,5,273.78
1,2018,8,2018-08,53888,370,25,28,145.64
2,2018,9,2018-09,111575,692,23,41,161.24
3,2018,10,2018-10,185028,998,23,53,185.40
4,2018,11,2018-11,263530,1285,25,59,205.08
5,2018,12,2018-12,227618,1177,19,89,193.39
6,2019,1,2019-01,294082,1771,21,90,166.05
7,2019,2,2019-02,272937,1757,20,90,155.34
8,2019,3,2019-03,279809,1841,21,90,151.99
9,2019,4,2019-04,262520,1657,19,90,158.43



TOP 10 MONTHS BY AVERAGE VISITS PER CENTRE-DAY


,Year,Month,YearMonth,TotalCustomerVisits,RecordedCentreDays,RecordedDates,ReportingCentres,AvgVisitsPerCentreDay
0,2018,7,2018-07,13689,50,13,5,273.78
4,2018,11,2018-11,263530,1285,25,59,205.08
5,2018,12,2018-12,227618,1177,19,89,193.39
51,2022,10,2022-10,331264,1753,23,91,188.97
3,2018,10,2018-10,185028,998,23,53,185.40
54,2023,1,2023-01,302354,1706,22,91,177.23
66,2024,1,2024-01,304001,1767,21,93,172.04
57,2023,4,2023-04,258205,1512,18,91,170.77
78,2025,1,2025-01,308926,1835,23,92,168.35
6,2019,1,2019-01,294082,1771,21,90,166.05



MONTHLY CUSTOMER TOTALS RECONCILED!


In [10]:

# STEP 23: BQ4 - Weekday Demand Analysis

conn = sqlite3.connect(db_path)

query_path = (
    project_root / "sql" /
    "07_bq4_weekday_demand.sql"
)

sql_query = query_path.read_text(encoding="utf-8")

weekday_result = pd.read_sql_query(
    sql_query,
    conn
)

print("BQ4: CUSTOMER DEMAND BY WEEKDAY")

display(weekday_result)

# Validate SQL totals against the processed dataset
expected_visits = data.loc[
    data["Date"].dt.year.between(2019, 2024),
    "CUSTOMERS"
].sum()

assert (
    weekday_result["TotalCustomerVisits"].sum()
    == expected_visits
)

print("\nWEEKDAY DEMAND RECONCILIATION PASSED!")

conn.close()


BQ4: CUSTOMER DEMAND BY WEEKDAY


,DayOfWeek,RecordedCentreDays,RecordedDates,ReportingCentres,TotalCustomerVisits,AvgVisitsPerCentreDay
0,Monday,24864,284,96,4159493,167.29
1,Tuesday,27042,308,97,4020147,148.66
2,Wednesday,27042,308,95,3757435,138.95
3,Thursday,26921,307,96,3958286,147.03
4,Friday,26114,304,95,4285998,164.13
5,Saturday,50,45,11,832,16.64
6,Sunday,3,3,2,3,1.00



WEEKDAY DEMAND RECONCILIATION PASSED!


In [14]:

# STEP 24: BQ5 - Monthly Waiting-Time Trends

conn = sqlite3.connect(db_path)

query_path = (
    project_root / "sql" /
    "08_bq5_monthly_wait_trends.sql"
)

sql_query = query_path.read_text(encoding="utf-8")

bq5_result = pd.read_sql_query(sql_query, conn)

print("BQ5: MONTHLY WAITING-TIME TRENDS")

display(bq5_result.head(12))

print("\nMOST RECENT 12 MONTHS IN ANALYSIS")

display(bq5_result.tail(12))

print("\nNUMBER OF MONTHS ANALYSED:")
print(len(bq5_result))

# Basic validation
assert bq5_result["YearMonth"].is_unique
assert bq5_result["WeightedWaitMinutes"].notna().all()

print("\nBQ5 BASIC VALIDATION PASSED!")

conn.close()


BQ5: MONTHLY WAITING-TIME TRENDS


,YearMonth,ReportingCentres,RecordedCentreDays,TotalVisits,WeightedWaitMinutes,PreviousMonthWait,ChangeMinutes,MoMChangePercentage
0,2019-01,21,441,126950,10.07,NaN,NaN,NaN
1,2019-02,21,417,117296,8.61,10.07,-1.46,-14.53
2,2019-03,21,434,118974,8.15,8.61,-0.46,-5.34
3,2019-04,21,399,112651,9.45,8.15,1.30,15.93
4,2019-05,21,458,127675,8.70,9.45,-0.75,-7.92
5,2019-06,21,417,115085,8.54,8.70,-0.16,-1.84
6,2019-07,21,481,133140,8.83,8.54,0.30,3.46
7,2019-08,21,453,127382,8.17,8.83,-0.67,-7.56
8,2019-09,21,441,121536,8.52,8.17,0.35,4.32
9,2019-10,21,462,126946,7.81,8.52,-0.71,-8.35



MOST RECENT 12 MONTHS IN ANALYSIS


,YearMonth,ReportingCentres,RecordedCentreDays,TotalVisits,WeightedWaitMinutes,PreviousMonthWait,ChangeMinutes,MoMChangePercentage
60,2024-01,22,438,121359,18.21,15.29,2.93,19.14
61,2024-02,22,459,119460,15.46,18.21,-2.75,-15.12
62,2024-03,22,436,114213,14.87,15.46,-0.59,-3.79
63,2024-04,22,438,117118,16.41,14.87,1.54,10.32
64,2024-05,22,480,126894,14.46,16.41,-1.94,-11.85
65,2024-06,22,436,114312,14.04,14.46,-0.42,-2.89
66,2024-07,22,504,130946,13.14,14.04,-0.91,-6.45
67,2024-08,22,470,120931,12.59,13.14,-0.55,-4.18
68,2024-09,22,462,117297,11.81,12.59,-0.77,-6.15
69,2024-10,22,484,123365,11.50,11.81,-0.32,-2.69



NUMBER OF MONTHS ANALYSED:
72

BQ5 BASIC VALIDATION PASSED!


In [15]:

# STEP 25: BQ5 - Annual Waiting-Time Trends

conn = sqlite3.connect(db_path)

query_path = (
    project_root / "sql" /
    "09_bq5_annual_wait_trends.sql"
)

sql_query = query_path.read_text(encoding="utf-8")

annual_result = pd.read_sql_query(
    sql_query,
    conn
)

print("BQ5: ANNUAL WAITING-TIME TRENDS")
display(annual_result)

# Validate annual reporting
assert len(annual_result) == 6

assert annual_result["ReportingCentres"].nunique() == 1

assert annual_result["WeightedWaitMinutes"].notna().all()

print("\nANNUAL TREND VALIDATION PASSED!")

conn.close()


BQ5: ANNUAL WAITING-TIME TRENDS


,Year,ReportingCentres,RecordedCentreDays,TotalVisits,WeightedWaitMinutes,YearlyChangeMinutes
0,2019,19,4744,1299514,8.44,NaN
1,2020,19,4734,1197628,7.05,-1.39
2,2021,19,4735,1204728,7.58,0.53
3,2022,19,4669,1251730,14.04,6.46
4,2023,19,4710,1228306,17.85,3.81
5,2024,19,4746,1200625,14.09,-3.76



ANNUAL TREND VALIDATION PASSED!


In [16]:

# STEP 26: BQ6 - Data Quality Summary

conn = sqlite3.connect(db_path)

query_path = (
    project_root / "sql" /
    "10_bq6_data_quality_summary.sql"
)

sql_query = query_path.read_text(encoding="utf-8")

bq6_result = pd.read_sql_query(sql_query, conn)

print("BQ6: DATA QUALITY SUMMARY BY CENTRE CATEGORY")

display(bq6_result)

# Validate against the original profiling findings
assert bq6_result["TotalCentres"].sum() == 99

assert bq6_result["TotalRecords"].sum() == 146219

assert bq6_result["ZeroWaitRecords"].sum() == 77277

assert bq6_result["LongWaitRecords"].sum() == 30

assert bq6_result["TotalCustomerVisits"].sum() == 22575820

print("\nBQ6 DATA QUALITY RECONCILIATION PASSED!")

conn.close()


BQ6: DATA QUALITY SUMMARY BY CENTRE CATEGORY


,CentreCategory,TotalCentres,TotalRecords,TotalCustomerVisits,ZeroWaitRecords,ZeroWaitPercentage,LongWaitRecords
0,All zero,39,55236,2471424,55236,100.00,0
1,Mixed,36,56115,10800456,22041,39.28,14
2,No zeros,24,34868,9303940,0,0.00,16



BQ6 DATA QUALITY RECONCILIATION PASSED!


In [17]:

# STEP 27: BQ6 - Reporting Gap Analysis

conn = sqlite3.connect(db_path)

query_path = (
    project_root / "sql" /
    "11_bq6_reporting_gaps.sql"
)

sql_query = query_path.read_text(encoding="utf-8")

gap_result = pd.read_sql_query(
    sql_query,
    conn
)

print("BQ6: TOP 10 CENTRES WITH POTENTIAL REPORTING GAPS")

display(gap_result)

# Basic validation
assert (
    gap_result["UnrecordedWeekdays"] >= 0
).all()

assert (
    gap_result["RecordedWeekdays"]
    + gap_result["UnrecordedWeekdays"]
    == gap_result["PotentialWeekdays"]
).all()

print("\nREPORTING GAP ANALYSIS VALIDATED!")

conn.close()


BQ6: TOP 10 CENTRES WITH POTENTIAL REPORTING GAPS


,CentreName,PotentialWeekdays,RecordedWeekdays,UnrecordedWeekdays,UnrecordedPercentage
0,Stradbroke Island CSC,1559,526,1033,66.26
1,Julia Creek QGAP,1554,807,747,48.07
2,Springsure QGAP,1555,847,708,45.53
3,Carseldine CSC 2,1471,1139,332,22.57
4,Richmond QGAP,1555,1268,287,18.46
5,Cunnamulla QGAP,1554,1298,256,16.47
6,Inglewood QGAP,1555,1311,244,15.69
7,Taroom QGAP,1555,1373,182,11.70
8,Texas QGAP,1555,1377,178,11.45
9,Charleville CSC,1560,1397,163,10.45



REPORTING GAP ANALYSIS VALIDATED!


In [18]:

# STEP 32: Prepare Power BI source files

import pandas as pd
import sqlite3
from pathlib import Path

# Locate the project database
project_root = next(
    (
        p for p in [Path.cwd(), Path.cwd().parent]
        if (p / "data/processed/service_performance.db").exists()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError("Project database not found")

processed = project_root / "data" / "processed"
db_path = processed / "service_performance.db"

export_folder = processed / "powerbi_exports"
export_folder.mkdir(parents=True, exist_ok=True)

# Export our three SQL tables
expected_rows = {
    "DimDate": 1800,
    "DimCentre": 99,
    "FactServicePerformance": 146219
}

with sqlite3.connect(db_path) as conn:
    for table, expected in expected_rows.items():

        table_data = pd.read_sql_query(
            f"SELECT * FROM {table}", conn
        )

        assert len(table_data) == expected

        table_data.to_csv(
            export_folder / f"{table}.csv",
            index=False
        )

        print(table, ":", len(table_data), "rows")

# Prepare forecast validation for Power BI
forecast = pd.read_csv(
    processed / "forecast_2024_validation.csv"
)

forecast = forecast.rename(columns={
    "ActualVisits": "ActualAvgVisitsPerCentreDay",
    "SeasonalNaiveForecast": "SeasonalNaiveAvgVisitsForecast",
    "HistoricalAverageForecast": "HistoricalAvgVisitsForecast"
})

forecast.to_csv(
    export_folder / "ForecastValidation2024.csv",
    index=False
)

# Export forecasting model comparison
model_comparison = pd.read_csv(
    processed / "forecast_model_comparison.csv"
)

model_comparison.to_csv(
    export_folder / "ForecastModelComparison.csv",
    index=False
)

print("\nPOWER BI SOURCE FILES READY!")
print("Export location:", export_folder)


DimDate : 1800 rows
DimCentre : 99 rows
FactServicePerformance : 146219 rows

POWER BI SOURCE FILES READY!
Export location: /Users/bps/Desktop/QLD-Service-Performance-Intelligence/data/processed/powerbi_exports
